# tqdm-postfix-metrics — worked example 1: Tracking accuracy and examples seen with tqdm postfix

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tqdm-postfix-metrics`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The `tqdm` progress bar can display live metrics alongside the bar by calling `pbar.set_postfix(**kwargs)`. You wrap any iterable with `tqdm(iterable, desc='...')` and call `set_postfix` on each step to update the display. Common training metrics like loss, accuracy, and examples seen make useful postfix fields. Recording each postfix call in a sidecar list lets tests inspect what was displayed.

## Worked solution

**Step 1 – Wrap the iterable.** We call `pbar = tqdm(enumerate(accs), desc='Eval', total=len(accs))`. Wrapping `enumerate(accs)` gives us both the step index and the value on each iteration, matching ARENA's inner-loop style.

**Step 2 – Initialize accumulators.** `examples_seen = 0` and `postfix_log = []` before the loop so they persist across iterations.

**Step 3 – Per-step update.** Inside the loop we add `batch_size` to `examples_seen`, then call `pbar.set_postfix(acc=f'{acc:.3f}', examples_seen=examples_seen)`. The `f-string` formats the float to three decimal places for a clean display. We also save the dict to `postfix_log`.

**Step 4 – Return.** Returning `postfix_log` gives the caller a record of exactly what was displayed at each step, which is what the test inspects.

In [ ]:
from tqdm import tqdm
import torch as t

def worked1_run_with_acc_postfix(accs, batch_size):
    """
    Wrap an accuracy sequence in tqdm and display live acc + examples_seen.
    Returns postfix_log: list of dicts that were set on the bar.
    """
    examples_seen = 0
    postfix_log = []
    pbar = tqdm(enumerate(accs), desc='Eval', total=len(accs))
    for step, acc in pbar:
        examples_seen += batch_size
        pf = dict(acc=f'{acc:.3f}', examples_seen=examples_seen)
        pbar.set_postfix(**pf)
        postfix_log.append(pf)
    return postfix_log

# Exercise it
t.manual_seed(0)
fake_accs = [float(v) for v in t.rand(5)]
log = worked1_run_with_acc_postfix(fake_accs, batch_size=32)
print('steps logged:', len(log))
print('last postfix:', log[-1])